# 02 · Is v0 more accurate than Savant? (parity)

The honest test of an xwOBA metric: does **this year's** number predict **next year's actual
wOBA**? We compare v0's model, public Savant xwOBA, and a naive "last year's actual wOBA"
baseline on 1,058 player-pairs with 100+ PA in both years. Loads `results/benchmark/`.

In [ ]:
# --- setup: find repo root, load helpers (run this first) ---
import sys, json
from pathlib import Path
import polars as pl
from IPython.display import Image, Markdown, display

pl.Config.set_tbl_rows(30)
here = Path.cwd()
ROOT = next((p for p in [here, *here.parents] if (p / "config.yaml").exists()), here)
sys.path.insert(0, str(ROOT))
RESULTS = ROOT / "results"

def jload(rel):
    return json.loads((RESULTS / rel).read_text())

def show_fig(rel, width=680):
    p = RESULTS / rel
    return Image(filename=str(p), width=width) if p.exists() else Markdown(f"_missing figure: {rel}_")

print("repo root:", ROOT)
print("results:  ", RESULTS, "(exists)" if RESULTS.exists() else "(MISSING)")

## Pooled predictive accuracy (Pearson r vs next-season actual wOBA)

In [ ]:
bm = jload("benchmark/benchmark_metrics.json")
p = bm["pooled"]
pl.DataFrame([
    {"predictor": "v0 model", "r": round(p["model"]["r"], 4), "rmse_calibrated": round(p["model"]["rmse_calibrated"], 4)},
    {"predictor": "Savant",   "r": round(p["savant"]["r"], 4), "rmse_calibrated": round(p["savant"]["rmse_calibrated"], 4)},
    {"predictor": "naive (last-yr wOBA)", "r": round(p["naive"]["r"], 4), "rmse_calibrated": round(p["naive"]["rmse_calibrated"], 4)},
])

## The gap: is v0 − Savant significantly different from zero?

In [ ]:
b = bm["model_vs_savant_bootstrap"]
print("n_pairs:", bm["n_pairs"])
print(f"r_model - r_savant = {b['r_model_minus_r_savant']:+.4f}")
print(f"95% CI: [{b['ci95'][0]:+.4f}, {b['ci95'][1]:+.4f}]")
print("fraction of bootstraps where model beats Savant:", b["frac_model_better"])
print("verdict:", bm["verdict"])
Markdown("**The CI straddles 0 → statistical parity. v0 does not beat Savant, but both clearly beat the naive baseline, so v0 is genuine xwOBA.**")

## Predictive-accuracy figure

In [ ]:
show_fig("benchmark/figures/predictive_accuracy.png")

## Why parity, and what it implies

The 3-feature model sits at **Savant's information ceiling** — tweaking it won't beat Savant;
only inputs Savant lacks (spray direction, batter handedness → a future **v1**) can.

**v1 target:** pooled r > 0.487 vs next-season actual wOBA, with a gap-CI excluding 0.

But accuracy was not the only goal — the project pivoted to *uncertainty* (how confident are
we in a hitter's number?) and *true-talent* estimates. → notebooks **03** and **04**.